In [ ]:
!nvidia-smi | head -12

# GPU benchmark — TokenVector.Inference vs PyTorch trên Colab GPU (T4)

GPT-2 124M, seq1024, fwd+bwd — **một protocol công bằng**:
cùng VM, cùng input token `[(i*37+11)%50257]`, seed 7, 2 thread,
**xen kẽ A/B 3 vòng mỗi bên, lấy median**.

**Chạy từ trên xuống** (reload rồi Run All là được):

- Cell [0/5]: nvidia-smi
- Cell [2/5]: cài .NET 10 SDK
- Cell [3/5]: clone (hoặc pull) repos + cài torch + LD_LIBRARY_PATH
- Cell [4/5]: build `.so` native (AVX2 + tkv_prof.c + tkv_attn_cuda.c)
- Cell [4.5/5]: build `tkv_torch_fused.so` (bind ATen CPU-oneDNN / **GPU với `TKV_TORCH_GPU=1`**)
- Cell [5/5]: `.tkv` **CPU native (TKV_TORCH_PATH= tắt ATen)** vs torch CPU 2 thread — fair
- Cell [6/5]: `.tkv` **GPU (TKV_TORCH_GPU=1, ATen qua CUDA)** vs torch CUDA — fair
- Cell [7/5]: TÁCH OP (diagnostic, TKV_PROF=1) — op nào ăn bao nhiêu
- Cell [ram]: peak RSS hai bên

> Loss `.tkv` ≠ torch (init scheme khác, FLOPs giống nhau) — chỉ so **thời gian**.
> Đúng đắn kiểm bằng fixtures/parity (`tv/tests/`), không phải bench.


In [ ]:
# [1/5] Install .NET 10 SDK (full SDK, includes runtime) + required libs
!apt-get update -qq && apt-get install -y -qq build-essential gcc libicu74 libssl3 libkrb5-3 zlib1g >/dev/null 2>&1 || apt-get install -y -qq build-essential gcc libicu-dev libkrb5-dev >/dev/null
!for u in https://dot.net/v1/dotnet-install.sh https://builds.dotnet.microsoft.com/dotnet/scripts/v1/dotnet-install.sh; do curl -fsSL --retry 5 --retry-delay 3 --connect-timeout 20 "$u" -o /tmp/dotnet-install.sh && break; echo "that bai: $u"; done; ls -l /tmp/dotnet-install.sh
!bash /tmp/dotnet-install.sh --channel 10.0 --install-dir /opt/dotnet
import os
os.environ["PATH"] = "/opt/dotnet:" + os.environ["PATH"]
!/opt/dotnet/dotnet --info | head -15


In [ ]:
# [3/5] Clone (hoac pull) repos + torch CUDA + LD_LIBRARY_PATH
!git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.git || (cd TokenVector && git pull --depth 1)
!git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.Inference.git || (cd TokenVector.Inference && git pull --depth 1)
!pip install -q nvidia-cublas-cu12 nvidia-cuda-runtime-cu12 2>&1 | tail -2
import os, glob
cublas_dir = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cublas/lib')), None)
cudart_dir = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cuda_runtime/lib')), None)
print('cublas_dir:', cublas_dir)
print('cudart_dir:', cudart_dir)
if cublas_dir:
    os.environ['LD_LIBRARY_PATH'] = cublas_dir + ':' + (cudart_dir or '') + ':' + os.environ.get('LD_LIBRARY_PATH', '')
    print('LD_LIBRARY_PATH =', os.environ['LD_LIBRARY_PATH'])
torch_dir = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/torch')), None)
print('torch:', torch_dir)
if torch_dir:
    os.environ['LD_LIBRARY_PATH'] = torch_dir + '/lib:' + os.environ['LD_LIBRARY_PATH']
    print('LD_LIBRARY_PATH =', os.environ['LD_LIBRARY_PATH'])


In [ ]:
# [4/5] Build .so native (C + AVX2 + CUDA dispatch)
!cd TokenVector/Release/tools/simd_native && gcc -O2 -mavx2 -mfma -shared -fPIC \
  tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c \
  -lm -lpthread -ldl -o tkv_simd_native.dll
!cp TokenVector/Release/tools/simd_native/tkv_simd_native.dll TokenVector.Inference/demo/tkv_simd_native.dll
!cp TokenVector/Release/tools/simd_native/tkv_simd_native.dll TokenVector.Inference/tkv_simd_native.dll
!cd TokenVector/Release/tools/simd_native && gcc -O2 -mavx2 -mfma -shared -fPIC tkv_step124.c -o tkv_step124.dll
!cp TokenVector/Release/tools/simd_native/tkv_step124.dll TokenVector.Inference/demo/tkv_step124.dll
print('Built .so')
import ctypes
lib = ctypes.CDLL('/content/TokenVector.Inference/demo/tkv_simd_native.dll')
lib.TkvCUDA_Available.restype = ctypes.c_int
print('TkvCUDA_Available() =', lib.TkvCUDA_Available(), '(1 = cuBLAS loaded, 0 = CPU fallback)')


In [ ]:
# [4.5/5] Build tkv_torch_fused.so (bind ATen CPU-oneDNN / GPU voi TKV_TORCH_GPU=1)
import subprocess, os, sys, glob
torch_dir = subprocess.check_output([sys.executable, '-c', 'import torch,os; print(os.path.dirname(torch.__file__))']).decode().strip()
print('torch:', torch_dir)
inc1 = torch_dir + '/include'
inc2 = torch_dir + '/include/torch/csrc/api/include'
libdir = torch_dir + '/lib'
build_cmd = ('cd /content/TokenVector/Release/tools/simd_native && '
    + 'g++ -O2 -shared -std=c++20 -fPIC torch_fused.cpp -o tkv_torch_fused.so '
    + '-I' + inc1 + ' -I' + inc2 + ' -L' + libdir + ' '
    + '-Wl,-rpath,' + libdir + ' '
    + '-ltorch -ltorch_cpu -lc10 -lc10_cuda -ltorch_cuda -Wl,-rpath-link,' + libdir + ' '
    + '2>&1 | tail -25')
print('--- build ---')
print(subprocess.run(build_cmd, shell=True, capture_output=True, text=True).stdout)
so_path = '/content/TokenVector/Release/tools/simd_native/tkv_torch_fused.so'
print('exists:', os.path.exists(so_path))
os.environ['TKV_TORCH_PATH'] = so_path
os.environ['LD_LIBRARY_PATH'] = libdir + ':' + os.environ.get('LD_LIBRARY_PATH', '')
import ctypes
lib = ctypes.CDLL(so_path)
print('ctypes load OK (tkv_torch_fused)')


In [ ]:
# [5/5] CPU: .tkv native (AVX2/BLAS) vs PyTorch CPU 2 thread — 2 cặp so sánh riêng
import subprocess, re, statistics as st
DOT = '$(command -v dotnet || echo /opt/dotnet/dotnet)'
def run_cmd(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800).stdout
def tkv_run():
    cmd = ('cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && '
           + 'TKV_TORCH_PATH= ' + DOT + ' demo/bseq.dll')
    out = run_cmd(cmd)
    m = re.search(r'fwd=([\d.]+) loss=([\d.]+) bwd=([\d.]+) opt=([\d.]+) copyback=([\d.]+) TONG=([\d.]+)', out)
    keys = ('fwd','ce','bwd','opt','copyback','total')
    return dict(zip(keys, map(float, m.groups())))
def torch_run():
    cmd = 'cd TokenVector.Inference && TKV_BENCH_DEVICE=cpu TKV_BENCH_THREADS=2 python3 benchmark/bench_torch.py e 1024'
    out = run_cmd(cmd)
    def g(pat):
        m = re.search(pat + r'[^\n]*?median=([\d.]+)', out)
        return float(m.group(1))
    return dict(zero=g('zero_grad'), fwd=g(r'fwd\(\+CE\)'), bwd=g('bwd seq1024'), opt=g('opt.step'), total=g('TOTAL'))
tk = []; tt = []
for r in range(3):
    if r % 2 == 0:
        a = tkv_run(); b = torch_run()
    else:
        b = torch_run(); a = tkv_run()
    tk.append(a); tt.append(b)
    print('vong %d: .tkv fwd=%.0f CE=%.0f bwd=%.0f opt=%.0f TONG=%.0f | torch fwd=%.0f bwd=%.0f opt=%.0f TONG=%.0f' % (
        r+1, a['fwd'], a['ce'], a['bwd'], a['opt'], a['total'], b['fwd'], b['bwd'], b['opt'], b['total']))
med = lambda xs: st.median(xs)
fb = med([x['fwd']+x['ce']+x['bwd'] for x in tk])
tot = med([x['total'] for x in tk])
tf = med([x['fwd'] for x in tt]) + med([x['bwd'] for x in tt])
ttot = med([x['total'] for x in tt])
print('\n==== MEDIAN (n=3, xen ke) ====')
print('.tkv fwd+CE+bwd = %.0f ms  | torch fwd+bwd = %.0f ms  | ty le = %.2fx' % (fb, tf, fb/tf))
print('.tkv TONG (full step) = %.0f ms  | torch TOTAL = %.0f ms  | ty le = %.2fx' % (tot, ttot, tot/ttot))


In [ ]:
# [6/5] GPU: .tkv GPU (TKV_TORCH_GPU=1) vs PyTorch CUDA — 2 cặp so sánh riêng
import subprocess, re, statistics as st
DOT = '$(command -v dotnet || echo /opt/dotnet/dotnet)'
def run_cmd(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800).stdout
def tkv_gpu():
    cmd = ('cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && '
           + 'TKV_TORCH_GPU=1 ' + DOT + ' demo/bseq.dll')
    out = run_cmd(cmd)
    m = re.search(r'fwd=([\d.]+) loss=([\d.]+) bwd=([\d.]+) opt=([\d.]+) copyback=([\d.]+) TONG=([\d.]+)', out)
    keys = ('fwd','ce','bwd','opt','copyback','total')
    return dict(zip(keys, map(float, m.groups())))
def torch_cuda():
    cmd = 'cd TokenVector.Inference && TKV_BENCH_DEVICE=cuda python3 benchmark/bench_torch.py d 1024'
    out = run_cmd(cmd)
    def g(pat):
        m = re.search(pat + r'[^\n]*?median=([\d.]+)', out)
        return float(m.group(1))
    return dict(fwdbwd=g(r'fwd\+bwd seq1024'), full=g(r'full step seq1024'))
tk = []; tt = []
for r in range(3):
    if r % 2 == 0:
        a = tkv_gpu(); b = torch_cuda()
    else:
        b = torch_cuda(); a = tkv_gpu()
    tk.append(a); tt.append(b)
    print('vong %d: .tkv fwd=%.0f CE=%.0f bwd=%.0f opt=%.0f TONG=%.0f | torch fwd+bwd=%.1f full=%.1f ms' % (
        r+1, a['fwd'], a['ce'], a['bwd'], a['opt'], a['total'], b['fwdbwd'], b['full']))
med = lambda xs: st.median(xs)
fb = med([x['fwd']+x['ce']+x['bwd'] for x in tk])
tot = med([x['total'] for x in tk])
tfb = med([x['fwdbwd'] for x in tt])
ttot = med([x['full'] for x in tt])
print('\n==== MEDIAN (n=3, xen ke) ====')
print('.tkv GPU fwd+CE+bwd = %.1f ms | torch fwd+bwd = %.1f ms | ty le = %.2fx' % (fb, tfb, fb/tfb))
print('.tkv GPU TONG (full step) = %.1f ms | torch full step = %.1f ms | ty le = %.2fx' % (tot, ttot, tot/ttot))


In [ ]:
# [7/5] TACH TUNG OP (diagnostic, KHONG phai benchmark) — so op native vs GPU
DOT = '$(command -v dotnet || echo /opt/dotnet/dotnet)'
def run(cmd):
    import subprocess, re
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800).stdout
    for line in out.splitlines():
        if 'BPHASE' in line or 'TKV_PROF' in line:
            print(line)
print('=== .tkv native (TKV_TORCH_PATH=) + TKV_PROF=1 ===')
run('cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && TKV_TORCH_PATH= TKV_PROF=1 ' + DOT + ' demo/bseq.dll')
print('=== .tkv GPU (TKV_TORCH_GPU=1) + TKV_PROF=1 ===')
run('cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && TKV_TORCH_GPU=1 TKV_PROF=1 ' + DOT + ' demo/bseq.dll')


In [ ]:
# [ram] Peak RAM (VmHWM): .tkv native vs torch CPU
!apt-get install -y -qq time >/dev/null 2>&1 || true
print('--- .tkv native (TKV_TORCH_PATH=), mode 0 seq1024, VmHWM ---')
!cd TokenVector.Inference && printf '1024\n0\n1\n' > tv/tests/_seq_cfg.txt && TKV_TORCH_PATH= /usr/bin/time -v $(command -v dotnet || echo /opt/dotnet/dotnet) demo/bseq.dll 2>&1 | grep -E 'BSEQ|Maximum resident'
print('--- PyTorch CPU (tier_d = fwd+bwd, T=2) ---')
!cd TokenVector.Inference && TKV_BENCH_DEVICE=cpu TKV_BENCH_THREADS=2 /usr/bin/time -v python3 benchmark/bench_torch.py d 1024 2>&1 | grep -E 'TOTAL|Maximum resident'
